# 🎯 06. Decision Threshold Optimization
### Amazon ML Challenge 2026: Multi-Source Business Entity Resolution

The challenge evaluation metric is **Macro-averaged $F_{0.5}$ per Source 1 entity**:
$$F_{0.5} = \frac{1.25 \times \text{Precision} \times \text{Recall}}{0.25 \times \text{Precision} + \text{Recall}}$$

Critical properties:
- **Precision is weighted 2× higher than Recall**: False merges are heavily penalized.
- **Singletons (no true matches):** Score $1.0$ if predicted blank `""`, and $0.0$ if any false merge is predicted.
In this notebook, we sweep the decision threshold across $[0.05, 0.95]$ to find the global $F_{0.5}$ maximum.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "amazon" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd

from src.utils.io import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")
thresh_path = PROJECT_ROOT / cfg["paths"]["thresholds_dir"] / "threshold.json"

if thresh_path.exists():
    with open(thresh_path, encoding="utf-8") as f:
        opt_res = json.load(f)
    print("Optimal Threshold Results:")
    print(f"  Best Threshold:             {opt_res['best_threshold']:.3f}")
    print(f"  Best Macro F0.5 Score:      {opt_res['best_score']:.4f}")
    print(f"  Overall Precision:          {opt_res['best_metrics'].get('overall_precision', 0):.4f}")
    print(f"  Overall Recall:             {opt_res['best_metrics'].get('overall_recall', 0):.4f}")
    print(f"  Singleton Accuracy:         {opt_res['best_metrics'].get('singletons_accuracy', 0):.4f}")

## 1. Threshold Sweep Trajectory
Inspect how Precision, Recall, and Macro $F_{0.5}$ vary across probability thresholds.

In [ ]:
if thresh_path.exists() and "sweep_history" in opt_res:
    df_sweep = pd.DataFrame(opt_res["sweep_history"])
    display(df_sweep.head(15))